# Lesson 09 — Capstone

Congratulations on getting here! 🎉

This lesson is different from the others. There's no new SQL to learn — you already know everything you need. Instead, we have one **multi-step problem** that uses the full toolkit, and we'll work through it together.

Plan on 30–40 minutes. Take it slow.

## Setup

In [ ]:
# Sign in to Google and connect to the Disney dataset.
# Run this once per Colab session.

from google.colab import auth
auth.authenticate_user()

from google.cloud.bigquery import magics
magics.context.project = "sql-with-disney"

print("✓ Ready! Run the query cells below.")

## What you've learned in 8 lessons

| Tool | What it does |
|------|--------------|
| `SELECT [columns]` | Pick columns |
| `FROM [table]` | Pick a table |
| `WHERE [condition]` | Filter rows |
| `ORDER BY [col] [DESC]` | Sort results |
| `LIMIT [n]` | Cap row count |
| `DISTINCT` | Remove duplicate rows |
| `COUNT/SUM/AVG/MIN/MAX` | Aggregate |
| `GROUP BY [col]` | One aggregate per group |
| `HAVING [agg condition]` | Filter on aggregates |
| `INNER JOIN` / `LEFT JOIN` | Combine tables |
| `CASE WHEN` | If/else logic in SQL |
| `EXTRACT`, `DATE_DIFF`, `CURRENT_DATE` | Date math |
| `IS NULL` / `IS NOT NULL` | Handle missing values |

That's the whole foundation. Real-world SQL queries combine these in different orders — there are very few "new" features beyond this list.

## The capstone problem

> **Question:** For each studio, what is its highest-grossing **non-action** movie released after 2000, and how does its Rotten Tomatoes score compare to that studio's overall average?

This single question uses: filtering (`WHERE`), aggregation (`MAX`, `AVG`), grouping (`GROUP BY`), and combining results across multiple aggregations.

Don't try to write the whole thing in one go. We're going to build it up in 4 steps.

## Step 1 — All non-action movies after 2000

Start with the simplest version of "the data we want to look at": every animation movie released after 2000.

Write a query that returns `title`, `studio`, `year`, `box_office_millions`, and `rt_score` for movies where `genre != 'Action'` and `year > 2000`.

Sort by box office, biggest first.

In [ ]:
# Your query here

### Step 1 solution

In [ ]:
%%bigquery
SELECT
  title,
  studio,
  year,
  box_office_millions,
  rt_score
FROM disney_lessons.movies
WHERE genre != 'Action'
  AND year > 2000
ORDER BY box_office_millions DESC

## Step 2 — Highest box office per studio (within those movies)

Now group by studio and find the top box office per group. Don't worry about getting the *title* of that highest-grossing movie yet — just the number.

In [ ]:
# Your query here

### Step 2 solution

In [ ]:
%%bigquery
SELECT
  studio,
  MAX(box_office_millions) AS top_box_office
FROM disney_lessons.movies
WHERE genre != 'Action'
  AND year > 2000
GROUP BY studio
ORDER BY top_box_office DESC

## Step 3 — Average score per studio (in the same filtered set)

Now compute the average Rotten Tomatoes score per studio, on the same filtered set.

In [ ]:
# Your query here

### Step 3 solution

In [ ]:
%%bigquery
SELECT
  studio,
  ROUND(AVG(rt_score), 1) AS avg_score
FROM disney_lessons.movies
WHERE genre != 'Action'
  AND year > 2000
GROUP BY studio

## Step 4 — Combine top box office + average score in one query

This is the trick: you can compute multiple aggregates per group in one `SELECT`. Combine Step 2 and Step 3:

In [ ]:
# Your query here

### Step 4 solution

In [ ]:
%%bigquery
SELECT
  studio,
  MAX(box_office_millions) AS top_box_office,
  ROUND(AVG(rt_score), 1) AS avg_score
FROM disney_lessons.movies
WHERE genre != 'Action'
  AND year > 2000
GROUP BY studio
ORDER BY top_box_office DESC

## Step 5 (the boss level) — Add the title of each studio's top movie

Right now you have `MAX(box_office_millions)` per studio — but no title. Adding the title is harder than it looks. You can't just do `SELECT title, MAX(box_office_millions)` because `title` isn't grouped — and there's no obvious way to ask "the title corresponding to the maximum."

There's a trick called **`QUALIFY`** (a GoogleSQL feature). It uses *window functions*, which are the next thing to learn after this course. For your interest only, here's what the full query looks like — don't worry about understanding `ROW_NUMBER` and `OVER` yet.

In [ ]:
%%bigquery
SELECT
  studio,
  title AS top_title,
  box_office_millions AS top_box_office,
  ROUND(AVG(rt_score) OVER (PARTITION BY studio), 1) AS studio_avg_score
FROM disney_lessons.movies
WHERE genre != 'Action'
  AND year > 2000
QUALIFY ROW_NUMBER() OVER (PARTITION BY studio ORDER BY box_office_millions DESC) = 1
ORDER BY top_box_office DESC

Run it and look at the result. For each studio, you see:
- Its top non-action post-2000 movie
- That movie's box office
- The studio's overall average score (in this same filtered set)

Don't try to memorize this. The point of showing it is: **once you've mastered the basics, there's a whole other layer (window functions, CTEs, recursive queries) that handles cases this query handles**. SQL gets surprisingly deep.

## What's next after this course?

You've learned the fundamentals. From here, the most useful next things to pick up are:

1. **CTEs (`WITH` clauses)** — break complex queries into named, readable chunks
2. **Window functions** — running totals, rankings, "top N per group" patterns
3. **String functions** — `LOWER`, `UPPER`, `CONCAT`, `REGEXP_CONTAINS`, etc.
4. **More date functions** — `DATE_TRUNC`, `DATE_ADD`, `FORMAT_DATE`
5. **The full BigQuery interface** — saved queries, scheduled queries, BigQuery Studio notebooks

When you write SQL in **other tools** (BigQuery Console, BigQuery Studio, internal data platforms), drop the `%%bigquery` line — everything else you've learned transfers as-is.

You're now able to ask real questions of real data on your own. That's a meaningful skill, and you got here in 9 lessons. 🎉

## Final read-only reminder

The Disney dataset stays read-only as long as you have the lesson permissions. When you eventually start querying real company data, the same SELECT/FROM/WHERE patterns apply — but you'll likely have access to many more tables. Ask your project lead or analytics team for the schema, and start exploring.